# Embedding Responses to Chalcogen Substitution

## Setup

In [ ]:
from setup import *

from matplotlib.lines import Line2D

## Compute Chalcogen Embeddings

In [ ]:
n_confs = 32
model_paths = {name: CHECKPOINTS[name] for name in ("nopre", "NT", "ANT")}

chalcogen_series = {  # series name (O analog unless noted): (bonding environment, SMILES template)
    "ethanol":          ("X-H",      "CC[XH]"),
    "phenol":           ("X-H",      "[XH]c1ccccc1"),
    "Ser / Cys / Sec":  ("X-H",      "N[C@@H](C[XH])C(=O)O"),
    "diethyl ether":    ("C-X-C",    "CC[X]CC"),
    "anisole":          ("C-X-C",    "C[X]c1ccccc1"),
    "Met (O / S / Se)": ("C-X-C",    "C[X]CC[C@H](N)C(=O)O"),
    "methyl acetate":   ("C-X-C",    "CC(=O)[X]C"),
    "diethyl peroxide": ("X-X",      "CC[X][X]CC"),
    "acetone":          ("C=X",      "CC(=[X])C"),
    "acetamide":        ("C=X",      "CC(N)=[X]"),
    "urea":             ("C=X",      "NC(N)=[X]"),
    "furan":            ("aromatic", "c1cc[x]c1"),
    "benzofuran":       ("aromatic", "c1ccc2[x]ccc2c1"),
    "dibenzofuran":     ("aromatic", "c1ccc2c(c1)[x]c1ccccc12"),
    "oxazole":          ("aromatic", "c1c[x]cn1"),
}

chalcogens = ("O", "S", "Se")
chalcogen_pairs = [("O", "S"), ("O", "Se"), ("S", "Se")]
n_elements = len(chalcogens)
element_index = {X: k for k, X in enumerate(chalcogens)}
chalcogen_smiles = [template.replace("[XH]", f"[{X}H]").replace("[X]", f"[{X}]").replace("[x]", f"[{X.lower()}]") for _, template in chalcogen_series.values() for X in chalcogens]
series_elements = [X for _ in chalcogen_series for X in chalcogens]
batches, mol_idx = batch_smiles_conformers(chalcogen_smiles, n_confs)
pca_coordinates, pair_similarities, neighbor_statistics = {}, {}, {}
for name, path in model_paths.items():
    embeddings = encode_batches(EncoderModel.load_model(path, device=DEVICE).eval(), batches).numpy()
    pca_coordinates[name] = compute_pca(embeddings)
    means = average_embeddings(embeddings, len(chalcogen_smiles))
    pair_similarities[name] = np.array([
        [means[n_elements * s + element_index[a]] @ means[n_elements * s + element_index[b]] for a, b in chalcogen_pairs]
        for s in range(len(chalcogen_series))])
    sims = means @ means.T
    np.fill_diagonal(sims, -np.inf)
    nearest = sims.argmax(1)
    deltas = means[element_index["S"]::n_elements] - means[element_index["O"]::n_elements]
    deltas /= np.linalg.norm(deltas, axis=1, keepdims=True)
    neighbor_statistics[name] = {
        "NN same series": np.mean(nearest // n_elements == np.arange(len(means)) // n_elements),
        "NN same chalcogen": np.mean(np.array(series_elements)[nearest] == np.array(series_elements)),
        "O->S direction, mean cos across series": (deltas @ deltas.T)[~np.eye(len(deltas), dtype=bool)].mean(),
    }

mean_similarities = {name: np.nanmean(values, axis=0) for name, values in pair_similarities.items()}

## Figure S9: Oxygen, Sulfur, and Selenium Analogues

In [ ]:
spectral = plt.get_cmap("Spectral")
parent_color = "#4d4d4d"
fig_width = 17.1
margin, gap, title_space = 0.05, 0.05, 0.3
chalcogen_colors = {"O": spectral(0.0), "S": spectral(0.33), "Se": spectral(0.9)}
environment_markers = {"X-H": "o", "C-X-C": "s", "X-X": "D", "C=X": "^", "aromatic": "h"}
series_env = [env for env, _ in chalcogen_series.values() for _ in chalcogens]
n_panels = len(model_paths)
side = 5.0  # cm, square; legend column is fig_width - legend_x = 1.9 cm
fig_h = round(margin + side + title_space + margin, 2)
to_fig = lambda x, y, w, h: [x / fig_width, y / fig_h, w / fig_width, h / fig_h]
fig = plt.figure(figsize=(fig_width * cm, fig_h * cm))
axes = [fig.add_axes(to_fig(margin + k * (side + gap), margin, side, side)) for k in range(n_panels)]
legend_x = margin + n_panels * (side + gap)
for ax, (name, pca) in zip(axes, pca_coordinates.items()):
    centroids = np.stack([pca[mol_idx == i].mean(0) for i in range(len(chalcogen_smiles))])
    for s in range(len(chalcogen_series)):
        ax.plot(*centroids[n_elements * s:n_elements * (s + 1)].T, color="0.75", linewidth=0.4, zorder=1)
    for i, (X, env) in enumerate(zip(series_elements, series_env)):
        pts = pca[mol_idx == i]
        ax.scatter(pts[:, 0], pts[:, 1], color=chalcogen_colors[X], marker=environment_markers[env], s=4 if X == "Se" else 10,
                   linewidths=0.15, edgecolors="white", zorder=3 + element_index[X])
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(MODEL_TITLES[name], fontsize=7, pad=2)
    for spine in ax.spines.values():
        spine.set_linewidth(0.25)

fig.legend(handles=[Line2D([0], [0], marker="o", linestyle="None", markersize=3.5, markerfacecolor=color, markeredgewidth=0, label=X)
                    for X, color in chalcogen_colors.items()],
           title="Chalcogen", title_fontsize=6, fontsize=5.5, frameon=False, loc="upper left",
           bbox_to_anchor=to_fig(legend_x, margin + side, 0, 0)[:2], handletextpad=0.3, labelspacing=0.4, borderaxespad=0, borderpad=0)
fig.legend(handles=[Line2D([0], [0], marker=marker, linestyle="None", markersize=3, markerfacecolor="none", markeredgecolor=parent_color, markeredgewidth=0.5, label=env)
                    for env, marker in environment_markers.items()],
           title="Environment", title_fontsize=6, fontsize=5.5, frameon=False, loc="lower left",
           bbox_to_anchor=to_fig(legend_x, margin, 0, 0)[:2], handletextpad=0.3, labelspacing=0.4, borderaxespad=0, borderpad=0)
fig.savefig(OUTPUT_DIR / "substitution_chalcogen_PCA.png", dpi=600)
plt.show()

## Within-Series Cosine Similarity Heatmap

In [ ]:
spectral = plt.get_cmap("Spectral")
parent_color = "#4d4d4d"
vmin, vmax = 0.3, 1.0
label_w, heat_w, heat_gap, env_w = 2.0, 1.45, 0.12, 1.0     # cm
top, bottom, row_h = 0.6, 0.35, 0.37                       # cm
heat_fig_w = label_w + len(pair_similarities) * heat_w + (len(pair_similarities) - 1) * heat_gap + env_w
heat_fig_h = top + bottom + row_h * len(chalcogen_series)
fig = plt.figure(figsize=(heat_fig_w * cm, heat_fig_h * cm))
environments = [env for env, _ in chalcogen_series.values()]
group_starts = [r for r in range(len(environments)) if r == 0 or environments[r] != environments[r - 1]] + [len(environments)]
for k, (name, sims) in enumerate(pair_similarities.items()):
    ax = fig.add_axes([(label_w + k * (heat_w + heat_gap)) / heat_fig_w, bottom / heat_fig_h, heat_w / heat_fig_w, row_h * len(chalcogen_series) / heat_fig_h])
    ax.imshow(sims, cmap=spectral, vmin=vmin, vmax=vmax, aspect="auto")
    for (r, c), v in np.ndenumerate(sims):
        red, green, blue, _ = spectral((v - vmin) / (vmax - vmin))
        ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=4, color="white" if 0.299 * red + 0.587 * green + 0.114 * blue < 0.5 else "black")
    ax.set_xticks(range(len(chalcogen_pairs)))
    ax.set_xticklabels([f"{a}-{b}" for a, b in chalcogen_pairs], fontsize=4.5)
    ax.set_yticks(range(len(chalcogen_series)))
    ax.set_yticklabels(list(chalcogen_series) if k == 0 else [], fontsize=4.5)
    ax.tick_params(length=0, pad=1.5)
    for spine in ax.spines.values():
        spine.set_visible(False)
    for start in group_starts[1:-1]:
        ax.axhline(start - 0.5, color="white", linewidth=1.2)
    ax.set_title("\n".join(MODEL_TITLES[name].rsplit(" ", 1)), fontsize=5.5, pad=3)  # two lines: the panels are narrow
for start, stop in zip(group_starts[:-1], group_starts[1:]):  # environment labels right of the last panel
    ax.text(len(chalcogen_pairs) - 0.5 + 0.15, (start + stop - 1) / 2, environments[start], fontsize=4.5, va="center", ha="left", color=parent_color)
fig.savefig(OUTPUT_DIR / "substitution_chalcogen_similarity.png", dpi=600)
plt.show()

## Within-Series Cosine Similarities

In [ ]:
display(pd.concat({
    name: pd.DataFrame(np.vstack([values, mean_similarities[name]]),
                       index=[*chalcogen_series, "mean"],
                       columns=[f"{a}-{b}" for a, b in chalcogen_pairs])
    for name, values in pair_similarities.items()
}, axis=1).rename(columns=MODEL_TITLES, level=0).round(3))

## Chalcogen Neighborhood Statistics

In [ ]:
display(pd.DataFrame(neighbor_statistics).rename(columns=MODEL_TITLES).round(3))